# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [4]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

print("Downloaded:", march_file)

Downloaded: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
import pandas as pd

df = pd.read_parquet(march_file)

print("Shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Shape: (9841378, 30)
Date range: 2026-03-01 to 2026-03-31


Signal 1 — Search volume (gsc_impressions)

Verdict: CONFIRMED

Reason: The March data contains a measurable distribution of impressions
across content rows, including higher-volume rows. This confirms that
search volume is a real observed signal that can be used for a
directional quick-win baseline.

In [6]:
# Signal 1: Search volume (impressions)

print("Total rows:", len(df))
print("Rows with impressions > 0:", (df["gsc_impressions"] > 0).sum())
print("Rows with impressions = 0:", (df["gsc_impressions"] == 0).sum())

print("\nImpression buckets:")

volume_buckets = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 0, 10, 100, 1000, float("inf")],
    labels=[
        "0",
        "1-10",
        "11-100",
        "101-1000",
        "1000+"
    ]
)

print(volume_buckets.value_counts().sort_index())

Total rows: 9841378
Rows with impressions > 0: 3611061
Rows with impressions = 0: 6230317

Impression buckets:
gsc_impressions
0           6230317
1-10        1531634
11-100      1445944
101-1000     601123
1000+         32360
Name: count, dtype: int64


Signal 2 — Click-through rate (CTR)

Verdict: CONFIRMED

Reason: CTR can be measured for rows with positive impressions using
observed GSC clicks and impressions. This provides a real signal for
identifying pages that receive search visibility but may have relatively
low click-through performance.

In [7]:
# Signal 2: CTR

ctr_df = df[df["gsc_impressions"] > 0].copy()

ctr_df["ctr"] = (
    ctr_df["gsc_clicks"] / ctr_df["gsc_impressions"]
)

print("Rows with measurable CTR:", len(ctr_df))

print("\nCTR statistics:")
print(ctr_df["ctr"].describe())

print("\nCTR buckets:")

ctr_buckets = pd.cut(
    ctr_df["ctr"],
    bins=[-0.001, 0.01, 0.03, 0.05, 0.10, 1.0, float("inf")],
    labels=[
        "<1%",
        "1-3%",
        "3-5%",
        "5-10%",
        "10-100%",
        "100%+"
    ]
)

print(ctr_buckets.value_counts().sort_index())

Rows with measurable CTR: 3611061

CTR statistics:
count    3.611061e+06
mean     3.080748e-03
std      3.009151e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: ctr, dtype: float64

CTR buckets:
ctr
<1%        3414297
1-3%        132328
3-5%         29724
5-10%        19098
10-100%      15614
100%+            0
Name: count, dtype: int64


In [8]:
# Part 2: Build ONE baseline action score

baseline = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks"
    ]
].copy()

# Calculate CTR only where impressions are positive
baseline["ctr"] = 0.0

mask = baseline["gsc_impressions"] > 0

baseline.loc[mask, "ctr"] = (
    baseline.loc[mask, "gsc_clicks"]
    / baseline.loc[mask, "gsc_impressions"]
)

# Score:
# Higher impressions + lower CTR = stronger CTR-fix opportunity
baseline["score"] = (
    baseline["gsc_impressions"] *
    (1 - baseline["ctr"])
)

# Reason code
baseline["reason_code"] = "CTR_FIX"

# Action label
baseline["action"] = "Improve CTR"

print("Baseline rows:", len(baseline))
print("\nScore statistics:")
print(baseline["score"].describe())

print("\nReason codes:")
print(baseline["reason_code"].value_counts())

print("\nActions:")
print(baseline["action"].value_counts())

Baseline rows: 9841378

Score statistics:
count    9.841378e+06
mean     2.843461e+01
std      1.554616e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.000000e+00
max      4.008300e+04
Name: score, dtype: float64

Reason codes:
reason_code
CTR_FIX    9841378
Name: count, dtype: int64

Actions:
action
Improve CTR    9841378
Name: count, dtype: int64


In [9]:
# Rank highest-priority opportunities first

baseline_ranked = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

print("Top 10 baseline recommendations:")

print(
    baseline_ranked[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Top 10 baseline recommendations:
            client_hash_id           content_hash_id  gsc_impressions  \
0  client_23a62021009f63c4  content_44f34c0a90047651            40084   
1  client_e547b89c05043229  content_eadb33b5df496f4a            39305   
2  client_62f4a7e64f5e0096  content_34a70fea29d15f24            39003   
3  client_e547b89c05043229  content_eadb33b5df496f4a            38436   
4  client_62f4a7e64f5e0096  content_945d6ff91386c817            37368   
5  client_e547b89c05043229  content_eadb33b5df496f4a            35404   
6  client_e547b89c05043229  content_eadb33b5df496f4a            34817   
7  client_e547b89c05043229  content_eadb33b5df496f4a            34606   
8  client_73cda7b4e4f265ea  content_fec55986a1868d62            33383   
9  client_e547b89c05043229  content_eadb33b5df496f4a            33571   

   gsc_clicks       ctr    score reason_code       action  
0           1  0.000025  40083.0     CTR_FIX  Improve CTR  
1         252  0.006411  39053.0     CTR_FI

In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_ranked.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows written:", len(baseline_ranked))

Saved: work/outputs/baseline_action_score.csv
Rows written: 9841378


### Baseline rule

I use a CTR-fix baseline. The score prioritizes content with higher
search impressions and lower observed CTR, because these rows have
search visibility but relatively weak click-through performance.

Score:
impressions × (1 − CTR)

Reason code:
CTR_FIX

Action:
Improve CTR

This is a directional decision-support baseline, not a causal prediction.

In [11]:
import pandas as pd
print("Pandas loaded:", pd.__version__)

Pandas loaded: 2.2.2


In [12]:
output_path = "work/outputs/baseline_action_score.csv"

print("Output path:", output_path)

Output path: work/outputs/baseline_action_score.csv


In [13]:
check = pd.read_csv(output_path)

print("CSV shape:", check.shape)

print("\nCSV columns:")
print(check.columns.tolist())

print("\nTop 10:")
print(check.head(10))

CSV shape: (9841378, 9)

CSV columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ctr', 'score', 'reason_code', 'action']

Top 10:
  report_date           client_hash_id           content_hash_id  \
0  2026-03-28  client_23a62021009f63c4  content_44f34c0a90047651   
1  2026-03-29  client_e547b89c05043229  content_eadb33b5df496f4a   
2  2026-03-04  client_62f4a7e64f5e0096  content_34a70fea29d15f24   
3  2026-03-28  client_e547b89c05043229  content_eadb33b5df496f4a   
4  2026-03-04  client_62f4a7e64f5e0096  content_945d6ff91386c817   
5  2026-03-30  client_e547b89c05043229  content_eadb33b5df496f4a   
6  2026-03-27  client_e547b89c05043229  content_eadb33b5df496f4a   
7  2026-03-31  client_e547b89c05043229  content_eadb33b5df496f4a   
8  2026-03-30  client_73cda7b4e4f265ea  content_fec55986a1868d62   
9  2026-03-24  client_e547b89c05043229  content_eadb33b5df496f4a   

   gsc_impressions  gsc_clicks       ctr    score reason_code       action  


### Top-10 review

1. Action: Improve CTR
   Why it is here: High search impressions with relatively low CTR.
   What would make it wrong: The page may already be performing well for
   its actual query mix, or impressions may come from low-intent searches.

2. Action: Improve CTR
   Why it is here: High search visibility and the rule assigns a high score.
   What would make it wrong: The observed CTR may not reflect the page's
   true opportunity because position and query intent are not included.

3. Action: Improve CTR
   Why it is here: High impressions combined with relatively low CTR.
   What would make it wrong: Search position or SERP features may explain
   the low CTR.

...

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [14]:
top10 = baseline_ranked.head(10).copy()

top10[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.000025,40083.0,CTR_FIX,Improve CTR
1,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,0.006411,39053.0,CTR_FIX,Improve CTR
2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.000051,39001.0,CTR_FIX,Improve CTR
3,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,0.007051,38165.0,CTR_FIX,Improve CTR
4,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,37368.0,CTR_FIX,Improve CTR
5,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,0.006355,35179.0,CTR_FIX,Improve CTR
6,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,0.006405,34594.0,CTR_FIX,Improve CTR
7,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,0.006791,34371.0,CTR_FIX,Improve CTR
8,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,33383.0,CTR_FIX,Improve CTR
9,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,0.006404,33356.0,CTR_FIX,Improve CTR


In [15]:
print("Future-window columns used: NONE")
print("Label-derived columns used: NONE")
print("Baseline uses only March observed signals.")

Future-window columns used: NONE
Label-derived columns used: NONE
Baseline uses only March observed signals.


### Weak picks

The lowest-ranked rows show where the baseline is least confident.
They generally have lower search visibility and/or a weaker opportunity
according to the CTR-fix score.

These rows should not automatically be treated as recommendations.
The baseline is a directional prioritization rule and does not establish
causality or guarantee that changing CTR will improve performance.

In [16]:
# Part 4: Inspect some weaker-ranked recommendations

weak_picks = baseline_ranked.tail(10).copy()

print("10 weakest-ranked rows:")

print(
    weak_picks[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

10 weakest-ranked rows:
                  client_hash_id           content_hash_id  gsc_impressions  \
9841368  client_cd12bcfd98942aa1  content_aa127fd3fbc9c079                0   
9841369  client_cd12bcfd98942aa1  content_80e57ebc41dc5064                0   
9841370  client_cd12bcfd98942aa1  content_04b8c57653a5853b                0   
9841371  client_cd12bcfd98942aa1  content_ae32465f79c723d2                0   
9841372  client_cd12bcfd98942aa1  content_2e717710eac18402                0   
9841373  client_cd12bcfd98942aa1  content_2471b8bd411ab29d                0   
9841374  client_cd12bcfd98942aa1  content_f7ec73329e35193a                0   
9841375  client_cd12bcfd98942aa1  content_97197f01f6175a7e                0   
9841376  client_cd12bcfd98942aa1  content_2ff1812bbc6ab0ba                0   
9841377  client_cd12bcfd98942aa1  content_d397a9901fce9f91                0   

         gsc_clicks  ctr  score reason_code       action  
9841368           0  0.0    0.0     CTR_FIX  Im

In [17]:
print("Scores sorted descending:",
      baseline_ranked["score"].is_monotonic_decreasing)

print("Highest score:", baseline_ranked["score"].max())
print("Lowest score:", baseline_ranked["score"].min())

Scores sorted descending: True
Highest score: 40083.0
Lowest score: 0.0


In [18]:
required_columns = [
    "score",
    "reason_code",
    "action"
]

print("Required columns present:")

for col in required_columns:
    print(col, "->", col in baseline_ranked.columns)

Required columns present:
score -> True
reason_code -> True
action -> True


In [19]:
# Final leakage check

future_columns = [
    "label",
    "target",
    "future",
    "next_month",
    "future_clicks",
    "future_impressions"
]

used_future_columns = [
    col for col in future_columns
    if col in baseline_ranked.columns
]

print("Future/label columns found:", used_future_columns)

print("\nFeatures used by baseline:")
print([
    "gsc_impressions",
    "gsc_clicks"
])

print("\nLeakage check:",
      "PASS" if len(used_future_columns) == 0 else "CHECK")

Future/label columns found: []

Features used by baseline:
['gsc_impressions', 'gsc_clicks']

Leakage check: PASS


In [20]:
import os

print("Notebook output file exists:",
      os.path.exists("work/outputs/baseline_action_score.csv"))

if os.path.exists("work/outputs/baseline_action_score.csv"):
    print(
        "File size:",
        round(
            os.path.getsize(
                "work/outputs/baseline_action_score.csv"
            ) / (1024 * 1024),
            2
        ),
        "MB"
    )

Notebook output file exists: True
File size: 875.15 MB


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.